# Contents:

Setup:
- Download tmol wheel (colab)
- Download a few input files
- Hello World of working with tmol: create a PoseStack and score it

Fundamentals:
- Initialize the default ParameterDatabase
- Load alternate block types
- Initialize a PackedBlockTypes object from the default database
- Initialize a PackedBlockTypes object from a custom database
- Create a CanonicalOrdering from a ParameterDatabase
- Create a PackedBlockTypes object from a subset of the block types in a ParameterDatabase

Input
- Initialize a single pose PoseStack from a PDB
- Initialize a pose stack from a subset of residues in a PDB
- Initialize a single pose PoseStack from a PDB file using biotite
- Initialize a single pose PoseStack from an .mmcif file (.cif)
- Initialize a single pose PoseStack from an OpenMM set of tensors
- Initialize a single pose PoseStack with missing residues
- Load a bunch of PoseStacks from different sources and then concatenate them to a single PoseStack
- Make many copies of the same single-pose PoseStack
- Add custom residue type
- Set the scoring parameters for a new residue type (elec, etc)
- Build an extended pose from sequence
- (Ligand features??)
- Create ligand block type from .params
- Create ligand block type from .mol2
- Create ligand block type from .cif that contains ligands
- Create BiotitePoseBuildContext with RefinedResidueTypes for repeat loading of ligand PDBs
- Load in multiple ligands and then process PDBs containing those ligands

Output:
- Write a single pose PoseStack to a .pdb file
- Write a multi-pose stack out as a multi-model PDB file
- Write a multi-pose stack out to separate PDB files
- Write a single pose PoseStack to an .mmcif file
- Write a rotamer set out as a multi-model PDB
- Write ligand-containing PoseStack out to .cif file using BiotitePoseBuildContext
- Write out ligand .params file

Kinematics:
- Create an N->C fold tree for a PoseStack
- Create a simple fold tree for a multi-chain PoseStack
- Create a simple fold tree for a PoseStack with missing residues
- Create a dandelion fold tree for a PoseStack
- Create a MoveMap that enables minimization for named torsions
- Create a MoveMap that enables backbone minimization for some residues but not all
- Apply a perturbation to the rigid-body DOFs between two chains
- Assign dihedral values to all the residues in a PoseStack and calculate the coordinates

Scoring:
- Create the default score function
- Create the default score function from a custom Database
- Create the soft-rep version of the score function
- Create an empty score function
- Turn on a few terms in a score function
- Turn off a term in a score function
- Score a PoseStack
- Score a PoseStack and back-propagate through the coordinates
- Score a PoseStack and return per-residue weighted energies
- Score a PoseStack and return per-residue weighted energies, weight them according to some principle, and then back-propagate the total energy.
- Add constraints to a PoseStack
- Add the same constraints to all the Poses in a PoseStack
- Add coordinate constraints to the current coordinates
- Alter the parameters for the cart-bonded energy function & rescore


Optimization
- Create a DunbrackSampler
- Add hydrogens
- Fill in side chains for a model that lacks them
- Perform fixed-sequence side-chain optimization: repack
- Repack with extra rotamers
- Add hydrogens and back-propagate
- Create a new PackerPalette subclass to handle logic of new block types
- Run minimization in double precision
- Perform cartesian minimization
- Perform kinematic minimization
- Relax a PoseStack w/ kinematic minimization
- Relax a PoseStack w/ cartesian minimization
- Relax structures generated one-at-a-time in batch format
- Idealize a structure from the PDB
- Idealize a structure from a dandelion
- Idealize just the backbone of a dandelion


In [ ]:
# For Colab
# install tmol directly from the github wheel;
!pip install https://github.com/uw-ipd/tmol/releases/download/v0.1.36/tmol-0.1.36+cu128torch2.10-cp312-cp312-linux_x86_64.whl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.5/95.5 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.0/58.0 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 78.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 307.5/307.5 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.9/155.9 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 94.9 MB/s eta 0:00:00


In [1]:
# download some structure files so we have something to work with
!wget -O 1ubq.pdb https://raw.githubusercontent.com/uw-ipd/tmol/refs/heads/master/tmol/tests/data/pdb/1ubq.pdb
!wget -O 1s78.pdb https://raw.githubusercontent.com/uw-ipd/tmol/refs/heads/master/tmol/tests/data/pdb/1s78.pdb
!wget -O 1qys.pdb https://raw.githubusercontent.com/uw-ipd/tmol/refs/heads/master/tmol/tests/data/pdb/1qys.pdb
!wget -O 1BL8.cif https://raw.githubusercontent.com/uw-ipd/tmol/refs/heads/master/tmol/tests/data/cif/1BL8.cif
!wget -O openfold_ubq_and_sumo.pt https://raw.githubusercontent.com/uw-ipd/tmol/refs/heads/master/tmol/tests/data/openfold/openfold_ubq_and_sumo.pt

--2026-07-21 15:54:07--  https://raw.githubusercontent.com/uw-ipd/tmol/refs/heads/master/tmol/tests/data/pdb/1ubq.pdb
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 99763 (97K) [text/plain]
Saving to: ‘1ubq.pdb’

1ubq.pdb            100%[===================>]  97.42K  --.-KB/s    in 0.1s    

2026-07-21 15:54:08 (742 KB/s) - ‘1ubq.pdb’ saved [99763/99763]

--2026-07-21 15:54:08--  https://raw.githubusercontent.com/uw-ipd/tmol/refs/heads/master/tmol/tests/data/pdb/1s78.pdb
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 615762 (60

In [2]:
# Make sure the files downloaded correctly; this should print the first 10 atoms
# from methionine in ubiquitin
!head 1ubq.pdb

ATOM      1  N   MET A   1      27.340  24.430   2.614  1.00  9.67           N  
ATOM      2  CA  MET A   1      26.266  25.413   2.842  1.00 10.38           C  
ATOM      3  C   MET A   1      26.913  26.639   3.531  1.00  9.62           C  
ATOM      4  O   MET A   1      27.886  26.463   4.263  1.00  9.62           O  
ATOM      5  CB  MET A   1      25.112  24.880   3.649  1.00 13.77           C  
ATOM      6  CG  MET A   1      25.353  24.860   5.134  1.00 16.29           C  
ATOM      7  SD  MET A   1      23.930  23.959   5.904  1.00 17.17           S  
ATOM      8  CE  MET A   1      24.447  23.984   7.620  1.00 16.11           C  
ATOM      9 1H   MET A   1      26.961  23.619   2.168  1.00  0.00           H  
ATOM     10 2H   MET A   1      28.043  24.834   2.029  1.00  0.00           H  


In [1]:
%env TMOL_USE_JIT=1

env: TMOL_USE_JIT=1


In [4]:
!export TMOL_USE_JIT=1

In [11]:
!echo $TMOL_USE_JIT

In [ ]:
import importlib

In [8]:
# importlib.reload(tmol)

NameError: name 'tmol' is not defined

In [3]:
import tmol

Loading compiled ops for tmol.kinematics...
1


In [5]:
print(tmol.__path__)

['/Users/leaverfa/GIT/tmol/tmol']


In [7]:
# The hello world of working with tmol:
# Load a PDB in from disk and score it

import tmol
import torch
import os

device = torch.device("cuda", torch.cuda.current_device()) if torch.cuda.is_available() else torch.device("cpu")

# Create a pose stack from a PDB.
pose_stack = tmol.pose_stack_from_pdb('1ubq.pdb', device=device)
# A PoseStack represents a batch of molecular systems (in this case, just a single structure - ubiquitin)
# PoseStacks are optimized for compactness for efficient processing on the GPU.
# Behind the scenes, pose_stack_from_pdb uses the default ParameterDatabase;
# which currently contains the parameters necessary to treat standard
# proteins, but little else. We will see more about the ParameterDatabase later.

# Create our score function.
sfxn = tmol.beta2016_score_function(device=device)
# This tmol score function is based on the Rosetta energy function.
# The score function is composed of many terms and weights for those terms.
# In this particular case, the score function terms and weights are set to
# match the beta2016_cart score function from Rosetta3. Again, we
# are relying on the default ParameterDatabase in the background.

# Create our scoring module.
scorer = sfxn.render_whole_pose_scoring_module(pose_stack)
# The scoring module is what does the actual score evaluation.
# This is separate from the ScoreFunction because it also needs details
# about the PoseStack being scored - mainly the Residue Types being used.
# The scoring module needs those Types because each score term must assemble
# compact tensors with the data necessary to score the Residue Types that are
# in use.

# Score the PoseStack and print the output.
print(scorer(pose_stack.coords))
# Return a tensor with the score of each pose in the stack (in this case, just 1 value)

tensor([235.1642], grad_fn=<SumBackward1>)


Fundamentals

In [ ]:
# - Initialize the default ParameterDatabase
param_db = tmol.ParameterDatabase.get_default()

In [ ]:
# - Load alternate block types
# TO DO: Kieran

In [ ]:
# - Initialize a PackedBlockTypes object from the default database
pbt = tmol.default_packed_block_types()

In [ ]:
# - Initialize a PackedBlockTypes object from a custom database
# TO DO

In [ ]:
# - Create a default CanonicalOrdering object
canonical_ordering = tmol.default_canonical_ordering()

In [ ]:
# - Create a CanonicalOrdering from a ParameterDatabase
# TO DO

In [ ]:
# - Create a PackedBlockTypes object from a subset of the block types in a ParameterDatabase
# TO DO

Input

In [ ]:
# - Initialize a single pose PoseStack from a PDB
pose_1ubq = tmol.pose_stack_from_pdb("1ubq.pdb", device=device)
assert pose_1ubq.max_n_blocks == 76

pose_1ubq 76


In [ ]:
# - Initialize a pose stack from a subset of residues in a PDB

# We have to tell the PDB reader that the first residue is not an
# N-terminus but merely is not connected to the residue that preceeds it
# and that the last residue is not a C-terminus.
res_not_connected = torch.zeros([1, 31, 2], dtype=torch.bool, device=device)
res_not_connected[0, 0, 0] = True
res_not_connected[0, 30, 1] = True

pose_1ubq_20to50 = tmol.pose_stack_from_pdb("1ubq.pdb", device=device, residue_start=20, residue_end=51, res_not_connected=res_not_connected)
assert pose_1ubq_20to50.inter_residue_connections[0,  0, 0, 0] == -1
assert pose_1ubq_20to50.inter_residue_connections[0, 30, 1, 0] == -1


In [8]:
# - Initialize a single pose PoseStack from a PDB file using biotite
import biotite.structure
import biotite.structure.io.pdb 

# TEMP
from tmol.io.pose_stack_from_biotite import pose_stack_from_biotite

bt_pdb_file = biotite.structure.io.pdb.PDBFile.read("1ubq.pdb")
bt_struct = bt_pdb_file.get_structure()
if isinstance(bt_struct, biotite.structure.AtomArrayStack):
    print("was atom array stack")
    bt_struct = bt_struct[0]

# TEMP: add pose_stack_from_biotite to API and modify this line
pose_1ubq_bt = pose_stack_from_biotite(bt_struct, device)

was atom array stack


/Users/leaverfa/GIT/tmol/tmol/pack/rotamer/build_rotamers.py:920: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  conf_dofs_kto[1:] = torch.tensor(


In [ ]:
# - Initialize a single pose PoseStack from an mmcif file (.cif)
import biotite.structure
from biotite.structure.io.pdbx import CIFFile, set_structure

# TEMP
from tmol.io.pose_stack_from_biotite import pose_stack_from_biotite

bt_struct = biotite.structure.io.load_structure(
    "1BL8.cif", extra_fields=["occupancy", "b_factor"]
)
if isinstance(bt_struct, biotite.structure.AtomArrayStack):
    print("was atom array stack")
    bt_struct = bt_struct[0]

# TEMP: add pose_stack_from_biotite to API and modify this line
pose_1bl8 = pose_stack_from_biotite(bt_struct, device)

/usr/local/lib/python3.12/dist-packages/tmol/pack/rotamer/build_rotamers.py:920: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  conf_dofs_kto[1:] = torch.tensor(


In [ ]:
# - Initialize a single pose PoseStack from an OpenFold set of tensors
# TO DO


In [ ]:
# - Initialize a single pose PoseStack with missing residues
# This particular structure has a few regions where the backbone is missing
pose_1s78 = tmol.pose_stack_from_pdb("1s78.pdb", device=device)
# TO DO: Assert that some residues are not bound to their upper / lower conns

In [ ]:
# - Load a bunch of PoseStacks from different sources and then concatenate them to a single PoseStack
# TEMP: Add PoseStackBuilder to API
from tmol.pose.pose_stack_builder import PoseStackBuilder
pose_stack_3 = PoseStackBuilder.from_poses([pose_1ubq, pose_1s78, pose_1bl8], device=device)

In [ ]:
# - Make many copies of the same single-pose PoseStack
# You can create a list of shallow copies of a single-pose PoseStack
# and the PoseStackBuilder will expand them into complete and fully
# independent poses.
ten_1ubqs = PoseStackBuilder.from_poses([pose_1ubq] * 10, device=device)
assert ten_1ubqs.n_poses == 10
assert ten_1ubqs.coords.shape[0] == 10

In [ ]:
# - Add custom residue type
# TO DO


In [ ]:
# - Set the scoring parameters for a new residue type (elec, etc)
# TO DO


In [ ]:
# - Build an extended pose from sequence
# TO DO

Output

In [ ]:
# - Write a single pose PoseStack to a .pdb file
tmol.write_pose_stack_pdb(pose_1ubq, "1ubq_out.pdb")
assert os.path.isfile("1ubq_out.pdb")

In [ ]:
# - Write a multi-pose stack out as a multi-model PDB file
tmol.write_pose_stack_pdb(ten_1ubqs, "ten_1ubqs.pdb")
assert os.path.isfile("ten_1ubqs.pdb")
def nlines_from_file(fname):
  with open(fname) as fid:
    lines = fid.readlines()
  return len(lines)
nlines_1ubq = nlines_from_file("1ubq.pdb")
nlines_ten_1ubqs = nlines_from_file("ten_1ubqs.pdb")
assert 10 * nlines_1ubq <= nlines_ten_1ubqs

In [ ]:
# - Write a multi-pose stack out to separate PDB files
for i in range(10):
  pose_i = ten_1ubqs.split(i)
  tmol.write_pose_stack_pdb(pose_i, f"1ubq_{i:04}.pdb")
for i in range(10):
  assert os.path.isfile(f"1ubq_{i:04}.pdb")

In [ ]:
# - Write a single pose PoseStack to an .mmcif file
# TO DO


In [ ]:
# - Write a rotamer set out as a multi-model PDB
# TO DO


Kinematics

In [ ]:
# - Create an N->C fold tree (a fold *forest* in tmol) for a PoseStack
import numpy

ff1_1ubq = tmol.FoldForest.reasonable_fold_forest(pose_1ubq)

# also valid: specify a root-jump to residue 0, and a polymer
# edge from 0 to the last residue in the pose.
edges = numpy.zeros((1, 2, 3), dtype=int)
edges[0, 0] = [tmol.EdgeType.root_jump, -1, 0]
edges[0, 1] = [tmol.EdgeType.polymer, 0, 75]
ff2_1ubq = tmol.FoldForest.from_edges(edges)

In [ ]:
# - Create a set of N->C fold trees for a multi-pose PoseStack
ff1_ps3 = tmol.FoldForest.reasonable_fold_forest(pose_stack_3)

edges = numpy.zeros((3, 2, 3), dtype=int)
edges[:, 0] = numpy.array([tmol.EdgeType.root_jump, -1, 0], dtype=int)[None, :]
n_res = pose_stack_3.n_res_per_pose().cpu().numpy()
edges[:, 1, 0] = tmol.EdgeType.polymer
edges[:, 1, 1] = -1
edges[:, 1, 2] = n_res
ff2_ps3 = tmol.FoldForest.from_edges

In [ ]:
# - Create a simple fold tree for a multi-chain PoseStack
# TO DO


In [ ]:
# - Create a simple fold tree for a PoseStack with missing residues
# TO DO

In [ ]:
# - Create a dandelion fold tree for a PoseStack
# TO DO

In [ ]:
# - Create a MoveMap that enables minimization for named torsions
# TO DO

In [ ]:
# - Create a MoveMap that enables backbone minimization for some residues but not all
# TO DO

In [ ]:
# - Apply a perturbation to the rigid-body DOFs between two chains
# TO DO

In [ ]:
# - Assign dihedral values to all the residues in a PoseStack and calculate the coordinates
# TO DO

Scoring

In [ ]:
# - Create the default score function
# TO DO

In [ ]:
# - Create the default score function from a custom Database
# TO DO

In [ ]:
# - Create the soft-rep version of the score function
# TO DO

In [ ]:
# - Create an empty score function
# TO DO

In [ ]:
# - Turn on a few terms in a score function
# TO DO

In [ ]:
# - Turn off a term in a score function
# TO DO

In [ ]:
# - Score a PoseStack
# TO DO

In [ ]:
# - Score a PoseStack and back-propagate through the coordinates
# TO DO

In [ ]:
# - Score a PoseStack and return per-residue weighted energies
# TO DO

In [ ]:
# - Score a PoseStack and return per-residue weighted energies, weight them according to some principle, and then back-propagate the total energy.
# TO DO

In [ ]:
# - Add constraints to a PoseStack
# TO DO

In [ ]:
# - Add the same constraints to all the Poses in a PoseStack
# TO DO

In [ ]:
# - Add coordinate constraints to the current coordinates
# TO DO

In [ ]:
# - Alter the parameters for the cart-bonded energy function & rescore
# TO DO

Optimization

In [ ]:
# - Create a DunbrackSampler
# TO DO

In [ ]:
# - Add hydrogens
# TO DO

In [ ]:
# - Fill in side chains for a model that lacks them
# TO DO

In [ ]:
# - Perform fixed-sequence side-chain optimization: repack
# TO DO

In [ ]:
# - Repack with extra rotamers
# TO DO

In [ ]:
# - Add hydrogens and back-propagate
# TO DO

In [ ]:
# - Create a new PackerPalette subclass to handle logic of new block types
# TO DO

In [ ]:
# - Run minimization in double precision
# TO DO

In [ ]:
# - Perform cartesian minimization
# TO DO

In [ ]:
# - Perform kinematic minimization
# TO DO

In [ ]:
# - Relax a PoseStack w/ kinematic minimization
# TO DO

In [ ]:
# - Relax a PoseStack w/ cartesian minimization
# TO DO

In [ ]:
# - Relax structures generated one-at-a-time in batch format
# TO DO

In [ ]:
# - Idealize a structure from the PDB
# TO DO

In [ ]:
# - Idealize a structure from a dandelion
# TO DO

In [ ]:
# - Idealize just the backbone of a dandelion
# TO DO